# Unified VLM Validation (Colab)

Этот ноутбук использует единый pipeline для нескольких датасетов.

Сейчас поддерживаются:
- `abo_150_expanded`
- `abo_physics_natural_bg_v2`

Что делает ноутбук:
1. Ставит зависимости.
2. Клонирует публичный репозиторий.
3. Загружает выбранный датасет через единый loader.
4. Валидирует выбранную VLM.
5. Сохраняет отчёты и при желании логирует в Comet.

Colab Secrets:
- `comet_api_key` — опционально
- `comet_workspace` — опционально
- `comet_project_name` — опционально


In [ ]:
# Dependencies (Colab-safe versions + one-time auto-restart)
import os
import sys
import subprocess
from pathlib import Path

MARKER = Path('/tmp/vlm_unified_validation_deps_ready')

if not MARKER.exists():
    print('Installing dependencies (first run)...')

    install_cmds = [
        [sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', 'numpy==2.1.3'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', '--upgrade-strategy', 'only-if-needed',
         'transformers>=4.49.0,<5.0.0', 'accelerate', 'bitsandbytes', 'sentencepiece'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', '--upgrade-strategy', 'only-if-needed',
         'pandas==2.2.2', 'pillow<12', 'pyyaml', 'boto3', 'rembg', 'onnxruntime', 'comet_ml'],
    ]

    for cmd in install_cmds:
        print('>>', ' '.join(cmd))
        subprocess.run(cmd, check=True)

    MARKER.write_text('ok')
    print('Dependencies installed. Restarting runtime now...')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies already installed in this runtime. Continue.')
            


In [ ]:
# Clone or update public repo in Colab
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Yaitco/VLM-2D-Physics-Boundaries.git"
WORKDIR = Path("/content/VLM-2D-Physics-Boundaries")

if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
else:
    print(f"Repo already exists: {WORKDIR}. Pulling latest...")
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only"], check=True)

%cd /content/VLM-2D-Physics-Boundaries


In [ ]:
import json
import os
import subprocess
from pathlib import Path

import pandas as pd

from scripts.abo150_vlm_validation import (
    MODEL_REGISTRY,
    get_available_variants,
    filter_property_specs,
    get_dataset_context,
    init_comet_experiment,
    is_known_value,
    load_protocol_property_specs,
    load_samples_for_dataset,
    resolve_protocol_schema_path,
    run_validation,
    save_report,
)

# Optional: load Hugging Face secrets from Colab if available
try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get('HF_TOKEN')
    hf_username = userdata.get('HF_USERNAME')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
    if hf_username:
        os.environ['HF_USERNAME'] = hf_username
except Exception:
    pass

# ---------- Dataset ----------
DATASET_NAME = 'abo_physics_natural_bg_v2'  # abo_physics_natural_bg_v2 | abo_150_expanded

# ---------- Protocol ----------
# Для abo_physics_natural_bg_v2: natural_bg_v2 | natural_bg_v2_main_material
# Для abo_150_expanded: narrow_core | full_expanded | pdf_compact
PROTOCOL_NAME = 'natural_bg_v2_main_material'

# ---------- Single-run config ----------
BASE_ADAPTER_MODEL_KEY = 'qwen3_vl_8b'
HUB_ADAPTER_OUTPUT_TAG = 'abo150_qwen3_main_material'
HUB_ADAPTER_REPO_ID = None  # e.g. 'your-hf-name/abo150_qwen3_main_material'
CUSTOM_MODEL_CONFIG_PATH = Path('outputs') / HUB_ADAPTER_OUTPUT_TAG / 'runtime_model_config.hub.json'
CUSTOM_MODEL_KEY = 'qwen3_vl_8b_qlora_main_material_hub'
PROPERTY_KEYS_OVERRIDE = None  # protocol already contains only intrinsic.main_material
PROPERTY_KEYS_MANIFEST_PATH = None  # e.g. 'outputs/abo150_qlora_main_material_dataset/manifest.json'

resolved_hub_adapter_repo_id = HUB_ADAPTER_REPO_ID
if resolved_hub_adapter_repo_id is None and os.getenv('HF_USERNAME'):
    resolved_hub_adapter_repo_id = f"{os.getenv('HF_USERNAME')}/{HUB_ADAPTER_OUTPUT_TAG}"

if not CUSTOM_MODEL_CONFIG_PATH.exists():
    if not resolved_hub_adapter_repo_id:
        raise ValueError(
            'runtime_model_config.hub.json not found. Set HUB_ADAPTER_REPO_ID explicitly or provide HF_USERNAME in Colab secrets.'
        )
    CUSTOM_MODEL_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    base_cfg = MODEL_REGISTRY[BASE_ADAPTER_MODEL_KEY]
    hub_runtime_cfg = {
        'backend': 'hf_chat',
        'model_id': f"{base_cfg['model_id']}+qlora@{resolved_hub_adapter_repo_id}",
        'base_model_id': base_cfg['model_id'],
        'adapter_path': resolved_hub_adapter_repo_id,
        'use_4bit': True,
        'max_new_tokens': int(base_cfg.get('max_new_tokens', 128)),
        'do_sample': bool(base_cfg.get('do_sample', False)),
        'max_prompt_batch_size': base_cfg.get('max_prompt_batch_size'),
    }
    CUSTOM_MODEL_CONFIG_PATH.write_text(
        json.dumps(hub_runtime_cfg, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

ACTIVE_MODEL_REGISTRY = dict(MODEL_REGISTRY)
ACTIVE_MODEL_REGISTRY[CUSTOM_MODEL_KEY] = json.loads(CUSTOM_MODEL_CONFIG_PATH.read_text(encoding='utf-8'))

SELECTED_MODEL = CUSTOM_MODEL_KEY
assert SELECTED_MODEL in ACTIVE_MODEL_REGISTRY, f'Unknown model key: {SELECTED_MODEL}'

# ---------- Batch grid config ----------
RUN_BATCH_GRID = False
BATCH_MODEL_KEYS = ['qwen3_vl_8b']
BATCH_VARIANTS = ['raw', 'mask_overlay', 'masked']
for model_key in BATCH_MODEL_KEYS:
    assert model_key in ACTIVE_MODEL_REGISTRY, f'Unknown model key in BATCH_MODEL_KEYS: {model_key}'

# ---------- Evaluation config ----------
MAX_SAMPLES = 100
RANDOM_SEED = 42
SAMPLE_IDS_PATH = None  # e.g. 'dataset/abo_150_expanded/splits/seed42_val50_train100/val_ids.txt'
EVAL_VARIANTS = ['raw']  # or ['raw', 'mask_overlay', 'masked']
META_OVERRIDE_PATH = None  # e.g. 'dataset/abo_physics_natural_bg_v2/review_outputs/segmentation_review_approved_meta.json'
MASK_FIELD = 'mask_path'  # mask_path | masks_hint | masks_hint_title
MASK_PREVIEW_FIELD = None  # None -> auto by MASK_FIELD, or e.g. masks_preview_hint
MASK_BACKGROUND_MODE = 'black'
SAVE_RAW_OUTPUT = True

INCLUDE_ONLY_GT_KNOWN = False
PROPERTY_BATCH_SIZE = 8
FEW_SHOT_K = 0
FEW_SHOT_SELECTION_MODE = 'fixed'

JSON_SUCCESS_THRESHOLD = 1.0
IMAGE_ID_SUCCESS_THRESHOLD = None

# ---------- Comet ----------
COMET_ENABLED = True
COMET_DEFAULT_PROJECT = 'vlm-physics-validation'

DATASET = get_dataset_context(DATASET_NAME)
REPORTS_DIR = DATASET.reports_dir or Path('reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('Dataset context:')
print(DATASET)
print('Single-run model:', SELECTED_MODEL)
print('Custom model config path:', CUSTOM_MODEL_CONFIG_PATH)
print('Resolved hub adapter repo id:', resolved_hub_adapter_repo_id)
print('Sample ids path:', SAMPLE_IDS_PATH)
print('Property keys override:', PROPERTY_KEYS_OVERRIDE)
print('Property keys manifest path:', PROPERTY_KEYS_MANIFEST_PATH)
print('Batch grid enabled:', RUN_BATCH_GRID)
print('Batch models:', BATCH_MODEL_KEYS)
print('Batch variants (requested):', BATCH_VARIANTS)
print('Meta override path:', META_OVERRIDE_PATH)
print('Mask field:', MASK_FIELD)
print('Mask preview field:', MASK_PREVIEW_FIELD)


In [ ]:
# Optional dataset-specific maintenance
if DATASET_NAME == 'abo_150_expanded':
    subprocess.run([
        'python', 'scripts/update_abo150_panel_paths.py',
        '--annotations-path', str(DATASET.annotations_path),
        '--panels-dir', str(DATASET.dataset_dir / 'selected_150_photos' / 'panels'),
        '--path-mode', 'relative',
    ], check=True)
    print('ABO150 panel_path updated.')
else:
    print(f'No dataset-specific maintenance needed for {DATASET_NAME}.')
            


In [ ]:
schema_path = resolve_protocol_schema_path(DATASET, PROTOCOL_NAME)
property_specs = load_protocol_property_specs(
    protocol_name=PROTOCOL_NAME,
    schema_path=schema_path,
)

selected_property_keys = PROPERTY_KEYS_OVERRIDE
if PROPERTY_KEYS_MANIFEST_PATH:
    manifest_cfg = json.loads(Path(PROPERTY_KEYS_MANIFEST_PATH).read_text(encoding='utf-8'))
    manifest_property_keys = manifest_cfg.get('property_keys')
    if not isinstance(manifest_property_keys, list) or not manifest_property_keys:
        raise ValueError(f"Expected non-empty property_keys in {PROPERTY_KEYS_MANIFEST_PATH}")
    selected_property_keys = [str(key) for key in manifest_property_keys]

if selected_property_keys:
    property_specs = filter_property_specs(property_specs, selected_property_keys)

samples = load_samples_for_dataset(
    dataset=DATASET,
    property_specs=property_specs,
    protocol_name=PROTOCOL_NAME,
    max_samples=MAX_SAMPLES,
    random_seed=RANDOM_SEED,
    mask_field=MASK_FIELD,
    mask_preview_field=MASK_PREVIEW_FIELD,
    meta_override_path=META_OVERRIDE_PATH,
    sample_ids_path=SAMPLE_IDS_PATH,
)

print(f'Loaded samples: {len(samples)}')
print(f'Protocol: {PROTOCOL_NAME}')
print(f'Evaluable properties: {len(property_specs)}')
print('Selected property keys:', list(property_specs.keys()))
print('First sample:')
print(json.dumps(
    {
        'image_id': samples[0]['image_id'],
        'path': samples[0]['path'],
        'mask_path': samples[0].get('mask_path'),
        'mask_preview_path': samples[0].get('mask_preview_path'),
        'mask_field': samples[0].get('mask_field'),
        'mask_preview_field': samples[0].get('mask_preview_field'),
        'known_properties': sum(
            1 for k, v in samples[0]['gt_properties'].items()
            if is_known_value(property_specs[k], v)
        ),
        'available_gt_keys': [
            k for k, v in samples[0]['gt_properties'].items()
            if is_known_value(property_specs[k], v)
        ],
    },
    ensure_ascii=False,
    indent=2,
))
            


In [ ]:
available_variants = get_available_variants(EVAL_VARIANTS, samples)
print('Running variants:', available_variants)

run_params = {
    'dataset_name': DATASET_NAME,
    'dataset_dir': str(DATASET.dataset_dir),
    'protocol_name': PROTOCOL_NAME,
    'selected_model_key': SELECTED_MODEL,
    'selected_model_id': ACTIVE_MODEL_REGISTRY[SELECTED_MODEL]['model_id'],
    'eval_variants_requested': ','.join(EVAL_VARIANTS),
    'eval_variants_actual': ','.join(available_variants),
    'sample_ids_path': SAMPLE_IDS_PATH,
    'meta_override_path': META_OVERRIDE_PATH,
    'mask_field': MASK_FIELD,
    'mask_preview_field': MASK_PREVIEW_FIELD,
    'mask_background_mode': MASK_BACKGROUND_MODE,
    'max_samples': MAX_SAMPLES,
    'random_seed': RANDOM_SEED,
    'include_only_gt_known': INCLUDE_ONLY_GT_KNOWN,
    'property_batch_size': PROPERTY_BATCH_SIZE,
    'few_shot_k': FEW_SHOT_K,
    'few_shot_selection_mode': FEW_SHOT_SELECTION_MODE,
    'json_success_threshold': JSON_SUCCESS_THRESHOLD,
    'image_id_success_threshold': IMAGE_ID_SUCCESS_THRESHOLD,
    'num_properties': len(property_specs),
    'selected_property_keys': '|'.join(property_specs.keys()),
    'property_keys_manifest_path': PROPERTY_KEYS_MANIFEST_PATH,
}

comet_experiment = init_comet_experiment(
    run_tag=f'{SELECTED_MODEL}_{DATASET_NAME}_{PROTOCOL_NAME}',
    run_params=run_params,
    enabled=COMET_ENABLED,
    default_project=COMET_DEFAULT_PROJECT,
)

try:
    variant_metrics = []

    for variant in available_variants:
        df_variant = run_validation(
            model_key=SELECTED_MODEL,
            samples=samples,
            property_specs=property_specs,
            model_registry=ACTIVE_MODEL_REGISTRY,
            variant=variant,
            property_batch_size=PROPERTY_BATCH_SIZE,
            include_only_gt_known=INCLUDE_ONLY_GT_KNOWN,
            few_shot_k=FEW_SHOT_K,
            few_shot_selection_mode=FEW_SHOT_SELECTION_MODE,
            mask_background_mode=MASK_BACKGROUND_MODE,
            save_raw_output=SAVE_RAW_OUTPUT,
            json_success_threshold=JSON_SUCCESS_THRESHOLD,
            image_id_success_threshold=IMAGE_ID_SUCCESS_THRESHOLD,
        )

        pm_variant, summary_variant = save_report(
            df=df_variant,
            model_key=SELECTED_MODEL,
            variant=variant,
            model_registry=ACTIVE_MODEL_REGISTRY,
            property_specs=property_specs,
            reports_dir=REPORTS_DIR / PROTOCOL_NAME,
            comet_experiment=comet_experiment,
        )

        pm_variant = pm_variant.copy()
        pm_variant['variant'] = variant
        variant_metrics.append(pm_variant)

    if variant_metrics:
        all_variant_metrics_df = pd.concat(variant_metrics, ignore_index=True)
        display(all_variant_metrics_df)

        if set(available_variants) >= {'raw', 'masked'}:
            pivot = all_variant_metrics_df.pivot(index='property', columns='variant', values='coverage_on_gt_known_pct')
            if 'raw' in pivot.columns and 'masked' in pivot.columns:
                pivot['delta_masked_minus_raw'] = pivot['masked'] - pivot['raw']
            print('\nCoverage comparison (masked vs raw):')
            display(pivot)
finally:
    if comet_experiment is not None:
        comet_experiment.end()


In [ ]:
# Optional: run separate experiments for each (model, variant) combination
# This is intentionally equivalent to running each experiment one by one.

if RUN_BATCH_GRID:
    batch_results = []

    batch_available_variants = get_available_variants(BATCH_VARIANTS, samples)
    print('Batch variants available in this dataset:', batch_available_variants)

    for model_key in BATCH_MODEL_KEYS:
        for variant in batch_available_variants:
            print(f'\n===== Running separate experiment: {model_key} [{variant}] =====')

            run_params = {
                'dataset_name': DATASET_NAME,
                'dataset_dir': str(DATASET.dataset_dir),
                'protocol_name': PROTOCOL_NAME,
                'selected_model_key': model_key,
                'selected_model_id': ACTIVE_MODEL_REGISTRY[model_key]['model_id'],
                'eval_variants_requested': ','.join(BATCH_VARIANTS),
                'eval_variants_actual': variant,
                'sample_ids_path': SAMPLE_IDS_PATH,
                'meta_override_path': META_OVERRIDE_PATH,
                'mask_field': MASK_FIELD,
                'mask_preview_field': MASK_PREVIEW_FIELD,
                'mask_background_mode': MASK_BACKGROUND_MODE,
                'max_samples': MAX_SAMPLES,
                'random_seed': RANDOM_SEED,
                'include_only_gt_known': INCLUDE_ONLY_GT_KNOWN,
                'property_batch_size': PROPERTY_BATCH_SIZE,
                'few_shot_k': FEW_SHOT_K,
                'few_shot_selection_mode': FEW_SHOT_SELECTION_MODE,
                'json_success_threshold': JSON_SUCCESS_THRESHOLD,
                'image_id_success_threshold': IMAGE_ID_SUCCESS_THRESHOLD,
                'num_properties': len(property_specs),
                'selected_property_keys': '|'.join(property_specs.keys()),
                'property_keys_manifest_path': PROPERTY_KEYS_MANIFEST_PATH,
            }

            experiment = init_comet_experiment(
                run_tag=f'{model_key}_{DATASET_NAME}_{PROTOCOL_NAME}_{variant}',
                run_params=run_params,
                enabled=COMET_ENABLED,
                default_project=COMET_DEFAULT_PROJECT,
            )

            try:
                df_variant = run_validation(
                    model_key=model_key,
                    samples=samples,
                    property_specs=property_specs,
                    model_registry=ACTIVE_MODEL_REGISTRY,
                    variant=variant,
                    property_batch_size=PROPERTY_BATCH_SIZE,
                    include_only_gt_known=INCLUDE_ONLY_GT_KNOWN,
                    few_shot_k=FEW_SHOT_K,
                    few_shot_selection_mode=FEW_SHOT_SELECTION_MODE,
                    mask_background_mode=MASK_BACKGROUND_MODE,
                    save_raw_output=SAVE_RAW_OUTPUT,
                    json_success_threshold=JSON_SUCCESS_THRESHOLD,
                    image_id_success_threshold=IMAGE_ID_SUCCESS_THRESHOLD,
                )

                _, summary_variant = save_report(
                    df=df_variant,
                    model_key=model_key,
                    variant=variant,
                    model_registry=ACTIVE_MODEL_REGISTRY,
                    property_specs=property_specs,
                    reports_dir=REPORTS_DIR / PROTOCOL_NAME,
                    comet_experiment=experiment,
                )

                batch_results.append({
                    'model_key': model_key,
                    'variant': variant,
                    'num_samples': summary_variant['num_samples'],
                    'report_dir': str((REPORTS_DIR / PROTOCOL_NAME / model_key / variant).resolve()),
                })
            finally:
                if experiment is not None:
                    experiment.end()

    if batch_results:
        print('\nBatch grid finished. Saved runs:')
        display(pd.DataFrame(batch_results))
else:
    print('Skip batch grid. Set RUN_BATCH_GRID=True to execute model x variant loops.')
